# MaNGA galaxies: EDA of spectra + morphology metadata (Google Colab)

This notebook explores two linked datasets:

| Dataset | Format | Contents |
|---|---|---|
| **Spectra** | ~16k `*.fits.gz` files in one `.zip` on Google Drive | One file per galaxy (for example `manga-10001-1901.fits.gz`). Columns: `WAVE`, `FLUX_1RE`, `FLUX_FoV`, `ERROR_1RE`, `ERROR_FoV` |
| **Metadata** | `morfologia.fits` (~16k rows) | `name`, `objra`, `objdec`, `Type`, `Ttype`, `nsa_z`, `LogMass`, `incl`, `PETRO_TH90` |

The spectrum file name matches the `name` column in the metadata.

**What the sample CSVs show** (checked before building the notebook):
* `WAVE` runs from about 3622 to 10351 Å on a linear 1.5 Å grid. It looks like the **observed frame** (the MaNGA range), so we shift to rest frame with `nsa_z`.
* Each file has **gaps**: masked pixels were dropped, so there are fewer rows than the full grid (~4,487).
* Some spectra end with placeholder pixels (`FLUX = 0` and a constant `ERROR`, such as 52). These are treated as invalid.
* `ERROR_FoV` is often much larger than `FLUX_FoV`. We look at this in the QC section.
* The metadata uses **sentinels** (`nsa_z = -9999`) and has **blank** values (`LogMass`, `incl`, `PETRO_TH90`).
* `Ttype` runs from −5 (E) through −2 (S0), 0 (S0a), 1–7 (Sa→Sd), plus `11` for `Type = "S"` (spiral with no sub-type).

### Notebook map
1. Setup & configuration
2. Load data from Google Drive (unzip spectra, read metadata)
3. Metadata EDA
4. Linking spectra ↔ metadata
5. Single-spectrum inspection
6. Bulk spectral QC + feature extraction (cached)
7. Spectral data-quality EDA (S/N, gaps, observed vs rest-frame masks)
8. Rest-frame stacking by morphological type
9. **Early- vs late-type galaxies: how the spectra and metadata relate to the classification**
10. Export

## 1. Setup

In [ ]:
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # astropy is usually preinstalled on Colab; umap-learn is optional (Section 9.6)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'astropy', 'umap-learn', 'pyarrow'], check=False)

In [ ]:
import os, re, glob, time, shutil, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import quad
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
from sklearn.inspection import permutation_importance

try:
    from astropy.io import fits
    from astropy.table import Table
except ImportError:
    fits = Table = None
    print('astropy not available -> FITS reading disabled (pip install astropy)')

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
_trapz = getattr(np, 'trapezoid', None) or np.trapz
print('numpy', np.__version__, '| pandas', pd.__version__)

### Configuration
**Edit the paths in this cell** to match where the files live in your Drive. Everything else uses these values.

In [ ]:
# ---------------- paths (EDIT ME) ----------------
DRIVE_MOUNT   = '/content/drive'
DRIVE_DIR     = '/content/drive/MyDrive/manga'            # folder in your Drive
ZIP_PATH      = f'{DRIVE_DIR}/espectros.zip'              # zip with ~16k *.fits.gz
METADATA_PATH = f'{DRIVE_DIR}/morfologia.fits'            # metadata table (.fits or .csv)

LOCAL_DIR   = '/content/manga_local'                      # fast local disk on the Colab VM
SPECTRA_DIR = f'{LOCAL_DIR}/spectra'                      # where the zip is extracted
CACHE_DIR   = f'{DRIVE_DIR}/eda_cache'                    # features/arrays cached here (persists across sessions)
SPEC_GLOBS  = ('*.fits.gz', '*.fits')                     # spectrum file patterns inside the zip

# ---------------- run options ----------------
MAX_GALAXIES  = None      # e.g. 500 for a quick dry run; None = all
N_JOBS        = -1        # parallel workers for spectrum processing
FORCE_RECOMPUTE = False   # ignore cached features
RANDOM_STATE  = 42

# ---------------- science options ----------------
SENTINELS = [-9999, -999.0, -99]          # values treated as missing in metadata
SPECTRA_IN_OBSERVED_FRAME = True          # WAVE is observed -> divide by (1+z) for rest frame
NOMINAL_DLAM  = 1.5                       # Å, observed pixel size seen in the CSVs
FULL_OBS_GRID = np.arange(3622.0, 10351.0 + 0.75, NOMINAL_DLAM)   # full nominal grid (~4487 px)
REST_GRID     = np.arange(3700.0, 8800.0, 2.0)                    # common rest-frame grid for stacking/PCA
NORM_WINDOW   = (5300.0, 5700.0)          # rest-frame window used to normalise spectra
MIN_BAND_COVERAGE = 0.7                   # min fraction of an index band that must have valid pixels
PIX_MIN_VALID = 0.5                       # PCA: keep rest-frame pixels valid in >= this fraction of galaxies
ROW_MIN_VALID = 0.5                       # PCA: keep galaxies with >= this fraction of those pixels valid

# Early/late definition (de Vaucouleurs T-type)
EARLY_TTYPE_MAX = 0      # E, S0-, S0, S0a  (Ttype <= 0) -> early
LATE_TTYPE_MIN, LATE_TTYPE_MAX = 1, 10   # Sa ... Im -> late
TTYPE_UNSPECIFIED_SPIRAL = 11            # 'S' without sub-type -> treated as late (set None to exclude)

H0, OM0 = 70.0, 0.3     # cosmology used for physical sizes

rng = np.random.default_rng(RANDOM_STATE)
def sample_n(d, n, seed=RANDOM_STATE):
    return d.sample(min(n, len(d)), random_state=seed)
os.makedirs(LOCAL_DIR, exist_ok=True)

## 2. Load data from Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
for p in [ZIP_PATH, METADATA_PATH]:
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p)
os.makedirs(CACHE_DIR, exist_ok=True)

### 2.1 Extract the spectra zip to local disk
Reading 16k small files straight from Drive is slow. So we copy the zip to the Colab VM once and extract it there. The extraction is skipped if it was already done in this session.

In [ ]:
def extract_spectra(zip_path, out_dir):
    marker = Path(out_dir) / '.extracted'
    if marker.exists():
        print('Already extracted ->', out_dir); return
    os.makedirs(out_dir, exist_ok=True)
    local_zip = Path(LOCAL_DIR) / Path(zip_path).name
    if not local_zip.exists():
        t0 = time.time(); shutil.copy(zip_path, local_zip)
        print(f'Copied zip to VM in {time.time()-t0:.0f}s ({local_zip.stat().st_size/1e9:.2f} GB)')
    t0 = time.time()
    if shutil.which('unzip'):
        subprocess.run(['unzip', '-q', '-o', str(local_zip), '-d', out_dir], check=True)
    else:
        with zipfile.ZipFile(local_zip) as zf:
            for m in tqdm(zf.infolist(), desc='unzip'):
                zf.extract(m, out_dir)
    marker.touch()
    print(f'Extracted in {time.time()-t0:.0f}s')

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names_in_zip = [n for n in zf.namelist() if not n.endswith('/')]
    print(f'{len(names_in_zip):,} entries in zip; examples:', names_in_zip[:3])
    extract_spectra(ZIP_PATH, SPECTRA_DIR)

spec_files = sorted({p for g in SPEC_GLOBS for p in glob.glob(f'{SPECTRA_DIR}/**/{g}', recursive=True)})
print(f'{len(spec_files):,} spectrum files found under {SPECTRA_DIR}')

### 2.2 Read the metadata table

In [ ]:
def load_metadata(path):
    if path.lower().endswith(('.fits', '.fits.gz', '.fit')):
        t = Table.read(path)
        multi = [c for c in t.colnames if len(t[c].shape) > 1]
        if multi: print('Dropping multidimensional columns:', multi); t.remove_columns(multi)
        df = t.to_pandas()
    else:
        df = pd.read_csv(path)
    # decode bytes / strip whitespace in string columns
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].apply(lambda v: v.decode() if isinstance(v, (bytes, bytearray)) else v)
            df[c] = df[c].astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    return df

meta_raw = load_metadata(METADATA_PATH)
print(meta_raw.shape)
meta_raw.head()

In [ ]:
meta = meta_raw.copy()
num_cols = ['objra', 'objdec', 'Ttype', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90']
for c in num_cols:
    meta[c] = pd.to_numeric(meta[c], errors='coerce')
n_sent = {c: int(meta[c].isin(SENTINELS).sum()) for c in num_cols}
print('Sentinel counts (set to NaN):', {k: v for k, v in n_sent.items() if v})
meta[num_cols] = meta[num_cols].mask(meta[num_cols].isin(SENTINELS))
meta.loc[meta['nsa_z'] <= 0, 'nsa_z'] = np.nan          # non-physical redshifts
meta['name'] = meta['name'].astype(str).str.strip()
meta['Type'] = meta['Type'].astype(str).str.strip()

# --- physical size: angular-diameter distance (flat LCDM) ---
C_KMS = 299792.458
_zg = np.linspace(0, 1, 2001)
_Dc = np.array([quad(lambda z: 1/np.sqrt(OM0*(1+z)**3 + 1-OM0), 0, zz)[0] for zz in _zg]) * C_KMS / H0   # Mpc
def kpc_per_arcsec(z):
    DA = np.interp(z, _zg, _Dc) / (1 + z)            # Mpc
    return DA * 1e3 * np.pi / (180 * 3600)
meta['R90_kpc'] = meta['PETRO_TH90'] * kpc_per_arcsec(meta['nsa_z'])

# --- morphology classes ---
def morph_class(tt):
    if pd.isna(tt): return 'unknown'
    if tt <= EARLY_TTYPE_MAX: return 'early'
    if LATE_TTYPE_MIN <= tt <= LATE_TTYPE_MAX: return 'late'
    if TTYPE_UNSPECIFIED_SPIRAL is not None and tt == TTYPE_UNSPECIFIED_SPIRAL: return 'late'
    return 'unknown'
def broad_type(row):
    t, tt = str(row['Type']), row['Ttype']
    if pd.isna(tt): return 'unknown'
    if tt <= -3 or t.startswith('E'): return 'E'
    if tt <= 0: return 'S0/S0a'
    return 'Spiral/Irr'
meta['morph_class'] = meta['Ttype'].apply(morph_class)
meta['broad_type']  = meta.apply(broad_type, axis=1)
meta['is_early']    = (meta['morph_class'] == 'early').astype(int)
meta.head()

## 3. Metadata EDA

### 3.1 Structure, missing values, duplicates

In [ ]:
print('Rows:', len(meta), '| unique names:', meta['name'].nunique())
dups = meta[meta['name'].duplicated(keep=False)].sort_values('name')
print('Duplicated names:', dups['name'].nunique())
if len(dups): display(dups.head(10))

miss = pd.DataFrame({'n_missing': meta.isna().sum(), 'pct_missing': 100*meta.isna().mean()}).sort_values('n_missing', ascending=False)
display(miss[miss.n_missing > 0].round(2))
display(meta[num_cols + ['R90_kpc']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(sample_n(meta[num_cols + ['R90_kpc']].isna(), 3000).T,
            cbar=False, cmap='Greys', ax=ax)
ax.set_title('Missing-value pattern (random 3000 rows; black = missing)'); ax.set_xticks([])
plt.tight_layout(); plt.show()

### 3.2 Morphology labels (`Type` and `Ttype`)

In [ ]:
ct = pd.crosstab(meta['Type'], meta['Ttype'])
order = meta.groupby('Type')['Ttype'].median().sort_values().index
display(ct.loc[order])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
vc = meta['Type'].value_counts().loc[order]
axes[0].bar(vc.index, vc.values, color='steelblue'); axes[0].tick_params(axis='x', rotation=90)
axes[0].set_title('Galaxies per Hubble Type (ordered by T-type)')
tc = meta['Ttype'].value_counts().sort_index()
axes[1].bar(tc.index.astype(int).astype(str), tc.values, color='indianred')
axes[1].set_title('Galaxies per T-type'); axes[1].set_xlabel('Ttype')
plt.tight_layout(); plt.show()

print(meta['morph_class'].value_counts(), '\n')
print(meta['broad_type'].value_counts())
# Bar-family breakdown (SA/SAB/SB) as an extra structural label
meta['bar'] = np.select([meta['Type'].str.startswith('SAB'), meta['Type'].str.startswith('SB'), meta['Type'].str.startswith('SA')],
                        ['SAB', 'SB', 'SA'], default='none/unspecified')
display(pd.crosstab(meta['bar'], meta['morph_class']))

### 3.3 Distributions of the numeric metadata

In [ ]:
plot_cols = ['nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc', 'Ttype']
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), plot_cols):
    v = meta[c].dropna()
    lo, hi = v.quantile([0.005, 0.995])
    sns.histplot(v.clip(lo, hi), bins=60, ax=ax, color='slategray')
    ax.set_title(f'{c}  (n={len(v):,}, median={v.median():.3g})')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
s = meta.dropna(subset=['objra', 'objdec'])
axes[0].scatter(s['objra'], s['objdec'], s=2, c=s['nsa_z'], cmap='viridis', vmax=s['nsa_z'].quantile(.99))
axes[0].invert_xaxis(); axes[0].set_xlabel('RA [deg]'); axes[0].set_ylabel('Dec [deg]'); axes[0].set_title('Sky footprint (colour = z)')
# Mass–redshift plane: selection effects (MaNGA targets brighter galaxies at higher z)
s = meta.dropna(subset=['nsa_z', 'LogMass'])
axes[1].scatter(s['nsa_z'], s['LogMass'], s=2, alpha=.4, c='k')
axes[1].set_xlabel('nsa_z'); axes[1].set_ylabel('log M*'); axes[1].set_title('Mass vs redshift (selection function)')
plt.tight_layout(); plt.show()

In [ ]:
corr_cols = ['Ttype', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc', 'objra', 'objdec']
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(meta[corr_cols].corr(method='spearman'), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Spearman correlation – metadata'); plt.tight_layout(); plt.show()

sp = sample_n(meta.dropna(subset=['LogMass', 'nsa_z', 'incl', 'R90_kpc']), 4000)
sns.pairplot(sp[['LogMass', 'nsa_z', 'incl', 'R90_kpc', 'broad_type']], hue='broad_type',
             plot_kws=dict(s=6, alpha=.4), corner=True, height=2.2)
plt.show()

## 4. Linking spectra ↔ metadata

In [ ]:
NAME_RE = re.compile(r'(manga-\d+-\d+)', re.I)
def name_from_path(p):
    m = NAME_RE.search(os.path.basename(p))
    return m.group(1).lower() if m else os.path.basename(p).split('.')[0]

files = pd.DataFrame({'path': spec_files})
files['name'] = files['path'].map(name_from_path)
files['size_kb'] = files['path'].map(lambda p: os.path.getsize(p) / 1024)
print('Duplicate spectrum names:', files['name'].duplicated().sum())
files = files.drop_duplicates('name')

meta['name_key'] = meta['name'].str.lower()
in_meta, in_files = set(meta['name_key']), set(files['name'])
print(f'Metadata rows with a spectrum : {len(in_meta & in_files):,}')
print(f'Metadata rows without spectrum: {len(in_meta - in_files):,}')
print(f'Spectra without metadata      : {len(in_files - in_meta):,}  e.g. {sorted(in_files - in_meta)[:5]}')

df = meta.merge(files.rename(columns={'name': 'name_key'}), on='name_key', how='inner')
if MAX_GALAXIES:
    df = sample_n(df, MAX_GALAXIES).reset_index(drop=True)
print('Working sample:', len(df))
fig, ax = plt.subplots(figsize=(6, 3)); sns.histplot(files['size_kb'], bins=60, ax=ax, color='steelblue'); ax.set_title('Spectrum file size [KB]'); plt.show()

## 5. Single-spectrum inspection

In [ ]:
SPEC_COLS = ['WAVE', 'FLUX_1RE', 'FLUX_FOV', 'ERROR_1RE', 'ERROR_FOV']

def read_spectrum(path):
    # Return a DataFrame with upper-cased canonical columns from .fits/.fits.gz/.csv.
    if path.lower().endswith('.csv'):
        d = pd.read_csv(path)
        d.columns = [c.strip().upper() for c in d.columns]
        return d.astype('float64')
    with fits.open(path, memmap=False) as hdul:
        tab = next((h for h in hdul if isinstance(h, (fits.BinTableHDU, fits.TableHDU))), None)
        if tab is not None:
            cols = {c.upper(): np.asarray(tab.data[c]) for c in tab.columns.names}
            if all(v.ndim == 2 and v.shape[0] == 1 for v in cols.values()):     # one row of vector columns
                cols = {k: v[0] for k, v in cols.items()}
            return pd.DataFrame({k: np.asarray(v, dtype='float64').ravel() for k, v in cols.items()})
        img = np.asarray(hdul[0].data, dtype='float64')                       # fallback: 2D image
        if img.ndim == 2 and 5 in img.shape:
            img = img if img.shape[1] == 5 else img.T
            return pd.DataFrame(img, columns=SPEC_COLS)
    raise ValueError(f'Unrecognised spectrum layout: {path}')

ex_path = df['path'].iloc[0]
if ex_path.lower().endswith(('.fits', '.fits.gz')):
    with fits.open(ex_path) as hdul:
        hdul.info()
        for h in hdul:
            print(f'--- {h.name} header (first 25 cards) ---'); print(repr(h.header[:25]))
sp = read_spectrum(ex_path)
print(sp.shape); display(sp.describe().T)

In [ ]:
def valid_mask(flux, err):
    return np.isfinite(flux) & np.isfinite(err) & (err > 0) & (flux != 0)

def plot_spectrum(row, ax=None, show_fov=True, rest=True):
    s = read_spectrum(row['path'])
    z = row['nsa_z'] if (rest and SPECTRA_IN_OBSERVED_FRAME and np.isfinite(row['nsa_z'])) else 0.0
    w = s['WAVE'].values / (1 + z)
    ax = ax or plt.gca()
    m = valid_mask(s['FLUX_1RE'].values, s['ERROR_1RE'].values)
    # break the line at gaps so masked regions are not drawn as straight segments
    f = np.where(m, s['FLUX_1RE'].values, np.nan)
    gap = np.r_[False, np.diff(s['WAVE'].values) > 1.5 * NOMINAL_DLAM]
    f[gap] = np.nan
    ax.plot(w, f, lw=.6, label='FLUX_1RE')
    if show_fov:
        ffov = np.where(valid_mask(s['FLUX_FOV'].values, s['ERROR_FOV'].values), s['FLUX_FOV'].values, np.nan); ffov[gap] = np.nan
        ax.plot(w, ffov, lw=.4, alpha=.6, label='FLUX_FoV')
    for lam, lab in [(3727, '[OII]'), (3934, 'CaK'), (3969, 'CaH'), (4000, 'D4000'), (4102, 'Hδ'), (4861, 'Hβ'),
                     (5007, '[OIII]'), (5175, 'Mgb'), (5893, 'NaD'), (6563, 'Hα')]:
        ax.axvline(lam, color='grey', lw=.4, ls=':'); ax.text(lam, 1.0, lab, transform=ax.get_xaxis_transform(), fontsize=7, rotation=90, va='top')
    ax.set_title(f"{row['name']}  Type={row['Type']}  T={row['Ttype']}  z={row['nsa_z']:.3f}  logM={row['LogMass']:.2f}", fontsize=9)
    ax.set_xlabel('rest-frame λ [Å]' if z else 'observed λ [Å]')
    return ax

# one example per broad type
ex = df.dropna(subset=['nsa_z']).groupby('broad_type').sample(1, random_state=RANDOM_STATE)
fig, axes = plt.subplots(len(ex), 1, figsize=(15, 3.2 * len(ex)))
for ax, (_, r) in zip(np.atleast_1d(axes), ex.iterrows()):
    plot_spectrum(r, ax); ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Error spectra & pixel S/N for the same example (1RE vs FoV)
r = ex.iloc[0]; s = read_spectrum(r['path'])
fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
for ap, c in [('1RE', 'C0'), ('FOV', 'C1')]:
    m = valid_mask(s[f'FLUX_{ap}'].values, s[f'ERROR_{ap}'].values)
    axes[0].plot(s['WAVE'][m], s[f'ERROR_{ap}'][m], lw=.6, c=c, label=f'ERROR_{ap}')
    axes[1].plot(s['WAVE'][m], (s[f'FLUX_{ap}'] / s[f'ERROR_{ap}'])[m], lw=.6, c=c, label=f'S/N {ap}')
axes[0].set_yscale('log'); axes[0].legend(); axes[1].legend(); axes[0].set_title('Error spectra'); axes[1].set_title('Pixel S/N')
plt.tight_layout(); plt.show()

## 6. Bulk QC and spectral feature extraction
We read every spectrum **once**. For each one we compute:
* **QC:** rows, wavelength range, valid fraction against the full 1.5 Å grid, number and size of gaps, zero/negative/non-finite pixels, and median S/N for both apertures.
* **Physically motivated indices** (measured in the rest frame on `FLUX_1RE`, and some also on `FLUX_FoV`):
  * `D4000n` (Balogh+99): the 4000 Å break, which traces the age of the stellar population.
  * `HdA` (Worthey & Ottaviani 97): Balmer absorption from A stars, which traces recent star formation over roughly the last Gyr.
  * Lick `Hb`, `Mgb`, `Fe5270`, `Fe5335`, `NaD`: these trace old populations and metallicity.
  * Emission EWs `EW_OII`, `EW_OIII`, `EW_Ha` (positive means emission): these trace current star formation and AGN activity.
  * `color_br`: a synthetic colour, $2.5\log_{10}(F_\nu^{6000-6200}/F_\nu^{4400-4600})$.
* A **normalised rest-frame spectrum** on `REST_GRID`, used for stacking and PCA.

The results are cached in `CACHE_DIR`, so this step only runs once.

In [ ]:
# blue continuum, central band, red continuum, kind ('abs' -> EW>0 for absorption; 'em' -> EW>0 for emission)
INDEX_DEFS = {
    'HdA':    ((4041.600, 4079.750), (4083.500, 4122.250), (4128.500, 4161.000), 'abs'),
    'Hb':     ((4827.875, 4847.875), (4847.875, 4876.625), (4876.625, 4891.625), 'abs'),
    'Mgb':    ((5142.625, 5161.375), (5160.125, 5192.625), (5191.375, 5206.375), 'abs'),
    'Fe5270': ((5233.150, 5248.150), (5245.650, 5285.650), (5285.650, 5318.150), 'abs'),
    'Fe5335': ((5304.625, 5315.875), (5312.125, 5352.125), (5353.375, 5363.375), 'abs'),
    'NaD':    ((5860.625, 5875.625), (5876.875, 5909.375), (5922.125, 5948.125), 'abs'),
    'EW_OII': ((3696.0, 3716.0), (3716.0, 3738.0), (3738.0, 3758.0), 'em'),
    'EW_OIII':((4978.0, 4998.0), (4998.0, 5015.0), (5015.0, 5035.0), 'em'),
    'EW_Ha':  ((6490.0, 6530.0), (6554.0, 6572.0), (6600.0, 6640.0), 'em'),   # avoids [NII]6548/6583
}

def _cov(w, lo, hi, dw):
    m = (w >= lo) & (w <= hi)
    return m, m.sum() * dw / (hi - lo)

def line_index(w, f, dw, blue, band, red, kind):
    mb, cb = _cov(w, *blue, dw); mr, cr = _cov(w, *red, dw); ml, cl = _cov(w, *band, dw)
    if min(cb, cr, cl) < MIN_BAND_COVERAGE: return np.nan
    fb, fr = np.median(f[mb]), np.median(f[mr])
    wb, wr = np.mean(blue), np.mean(red)
    cont = fb + (fr - fb) * (w[ml] - wb) / (wr - wb)
    if np.any(cont <= 0): return np.nan
    ratio = f[ml] / cont
    return float(np.sum((1 - ratio) if kind == 'abs' else (ratio - 1)) * dw)

def mean_fnu(w, f, lo, hi, dw):
    m, c = _cov(w, lo, hi, dw)
    return np.mean(f[m] * w[m] ** 2) if c >= MIN_BAND_COVERAGE else np.nan

def d4000n(w, f, dw):
    b, r = mean_fnu(w, f, 3850, 3950, dw), mean_fnu(w, f, 4000, 4100, dw)
    return r / b if (b and b > 0) else np.nan

def resample_rest(w, f, grid, dw):
    if len(w) < 10: return np.full(len(grid), np.nan, np.float32)
    out = np.interp(grid, w, f, left=np.nan, right=np.nan)
    idx = np.clip(np.searchsorted(w, grid), 1, len(w) - 1)
    dist = np.minimum(np.abs(grid - w[idx - 1]), np.abs(w[idx] - grid))
    out[dist > 1.5 * dw] = np.nan                          # do not bridge masked gaps
    return out.astype(np.float32)

def process_one(path, z):
    rec = {'path': path, 'read_error': None}
    try:
        s = read_spectrum(path)
    except Exception as e:
        rec['read_error'] = repr(e)[:200]; return rec, None
    wobs = s['WAVE'].values
    rec.update(n_rows=len(s), wave_min=np.nanmin(wobs), wave_max=np.nanmax(wobs),
               n_nonfinite=int((~np.isfinite(s[SPEC_COLS[1:]].values)).any(axis=1).sum()))
    dwo = np.diff(wobs)
    gaps = dwo[dwo > 1.5 * NOMINAL_DLAM]
    rec.update(n_gaps=len(gaps), max_gap_A=float(gaps.max()) if len(gaps) else 0.0,
               missing_A=float((gaps - NOMINAL_DLAM).sum()), non_monotonic=int((dwo <= 0).sum()))
    zz = z if (SPECTRA_IN_OBSERVED_FRAME and np.isfinite(z)) else 0.0
    out_arr = None
    for ap in ['1RE', 'FOV']:
        f, e = s[f'FLUX_{ap}'].values, s[f'ERROR_{ap}'].values
        m = valid_mask(f, e)
        rec[f'frac_valid_{ap}'] = m.sum() / len(FULL_OBS_GRID)
        rec[f'n_zero_{ap}'] = int((f == 0).sum()); rec[f'n_neg_{ap}'] = int((f < 0).sum())
        rec[f'snr_{ap}'] = float(np.nanmedian(f[m] / e[m])) if m.any() else np.nan
        rec[f'medflux_{ap}'] = float(np.nanmedian(f[m])) if m.any() else np.nan
        if not np.isfinite(z) and SPECTRA_IN_OBSERVED_FRAME:
            continue
        w, fv = wobs[m] / (1 + zz), f[m]
        dw = NOMINAL_DLAM / (1 + zz)
        rec[f'D4000n_{ap}'] = d4000n(w, fv, dw)
        rec[f'EW_Ha_{ap}'] = line_index(w, fv, dw, *INDEX_DEFS['EW_Ha'])
        if ap == '1RE':
            for k, d in INDEX_DEFS.items():
                rec[k] = line_index(w, fv, dw, *d)
            fb, fr = mean_fnu(w, fv, 4400, 4600, dw), mean_fnu(w, fv, 6000, 6200, dw)
            rec['color_br'] = 2.5 * np.log10(fr / fb) if (fb > 0 and fr > 0) else np.nan
            mn, cn = _cov(w, *NORM_WINDOW, dw)
            norm = np.median(fv[mn]) if cn > 0.5 else np.nan
            rec['norm_flux'] = norm
            rec['snr_norm'] = float(np.median(fv[mn] / e[m][mn])) if cn > 0.5 else np.nan
            if np.isfinite(norm) and norm > 0:
                out_arr = resample_rest(w, fv / norm, REST_GRID, dw)
    return rec, out_arr

r0, a0 = process_one(df['path'].iloc[0], df['nsa_z'].iloc[0])
pd.Series(r0)

In [ ]:
feat_cache = Path(CACHE_DIR) / f'spectral_features_{len(df)}.parquet'
arr_cache  = Path(CACHE_DIR) / f'rest_spectra_{len(df)}.npy'

if feat_cache.exists() and arr_cache.exists() and not FORCE_RECOMPUTE:
    feats = pd.read_parquet(feat_cache); REST = np.load(arr_cache)
    print('Loaded cache:', feat_cache)
else:
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS, batch_size=64)(
        delayed(process_one)(p, z) for p, z in tqdm(list(zip(df['path'], df['nsa_z'])), desc='spectra'))
    feats = pd.DataFrame([r for r, _ in res])
    REST = np.vstack([a if a is not None else np.full(len(REST_GRID), np.nan, np.float32) for _, a in res])
    print(f'Processed {len(res):,} spectra in {(time.time()-t0)/60:.1f} min')
    feats.to_parquet(feat_cache); np.save(arr_cache, REST)

data = df.merge(feats, on='path', how='left')
assert len(data) == len(REST)
print(data.shape, REST.shape)
print('Read errors:', data['read_error'].notna().sum())
display(data.loc[data['read_error'].notna(), ['name', 'read_error']].head())

## 7. Spectral data-quality EDA

In [ ]:
qc_cols = ['n_rows', 'wave_min', 'wave_max', 'frac_valid_1RE', 'frac_valid_FOV', 'n_gaps', 'max_gap_A', 'missing_A',
           'n_zero_1RE', 'n_neg_1RE', 'n_nonfinite', 'non_monotonic', 'snr_1RE', 'snr_FOV', 'snr_norm']
display(data[qc_cols].describe(percentiles=[.01, .05, .5, .95, .99]).T.round(3))

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, c in zip(axes.ravel(), ['n_rows', 'frac_valid_1RE', 'n_gaps', 'max_gap_A', 'snr_1RE', 'snr_FOV', 'snr_norm', 'n_zero_1RE']):
    v = data[c].replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(v.clip(*v.quantile([0.005, 0.995])), bins=60, ax=ax, color='steelblue'); ax.set_title(c)
plt.tight_layout(); plt.show()

**About the FoV errors.** In the sample CSVs, `ERROR_FoV` is almost constant and is often larger than `FLUX_FoV`, so the per-pixel S/N is below 1. Meanwhile `FLUX_FoV` is smooth and much brighter than `FLUX_1RE`. This suggests the FoV error is either propagated in a very conservative way or stored in different units (for example, a variance-like quantity). The cell below compares the empirical noise (the scatter between neighbouring pixels) with the reported error for each aperture, to check whether the errors can be trusted.

In [ ]:
def empirical_vs_reported(path):
    s = read_spectrum(path); out = {}
    for ap in ['1RE', 'FOV']:
        f, e = s[f'FLUX_{ap}'].values, s[f'ERROR_{ap}'].values
        m = valid_mask(f, e); fm = f[m]
        emp = 1.4826 * np.median(np.abs(np.diff(fm) - np.median(np.diff(fm)))) / np.sqrt(2)   # robust pixel-to-pixel noise
        out[f'ratio_{ap}'] = np.median(e[m]) / emp if emp > 0 else np.nan
    return out
sub = sample_n(data, 400)
evr = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(empirical_vs_reported)(p) for p in sub['path']))
display(evr.describe().T.round(2))
print('ratio ≈ 1 -> reported error matches the scatter; >>1 -> error overestimated / different units')

### 7.1 Are the gaps fixed in the observed frame or the rest frame?
If the masked pixels line up in the **observed** frame, they come from sky lines, detector artefacts or the dichroic. If they line up in the **rest** frame, someone may have masked emission lines on purpose, and that would bias `EW_Ha`/`EW_OII`. Both cases matter for interpretation.

In [ ]:
def presence_on_grid(path, z):
    s = read_spectrum(path); w = s['WAVE'].values
    m = valid_mask(s['FLUX_1RE'].values, s['ERROR_1RE'].values)
    obs = np.zeros(len(FULL_OBS_GRID), bool)
    idx = np.clip(np.round((w[m] - FULL_OBS_GRID[0]) / NOMINAL_DLAM).astype(int), 0, len(FULL_OBS_GRID) - 1)
    obs[idx] = True
    rest_w = FULL_OBS_GRID / (1 + z)
    rest = np.interp(REST_GRID, rest_w, obs.astype(float), left=np.nan, right=np.nan)
    return obs, rest

sub = sample_n(data.dropna(subset=['nsa_z']), 1500)
pres = Parallel(n_jobs=N_JOBS)(delayed(presence_on_grid)(p, z) for p, z in zip(sub['path'], sub['nsa_z']))
obs_missing  = 1 - np.mean([o for o, _ in pres], axis=0)
rest_missing = 1 - np.nanmean([r for _, r in pres], axis=0)

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharey=True)
axes[0].plot(FULL_OBS_GRID, obs_missing, lw=.7); axes[0].set_title('Fraction of spectra masked at each OBSERVED wavelength')
axes[1].plot(REST_GRID, rest_missing, lw=.7, c='C3'); axes[1].set_title('Fraction of spectra masked at each REST wavelength')
for lam in [3727, 4861, 5007, 6563, 6717]:
    axes[1].axvline(lam, c='grey', ls=':', lw=.6)
for lam in [5577, 6300, 5893]:
    axes[0].axvline(lam, c='grey', ls=':', lw=.6)
axes[1].set_xlabel('λ [Å]'); plt.tight_layout(); plt.show()
print('Sharp spikes in the top panel -> observed-frame (sky/instrument) masks; spikes at emission lines in the bottom panel -> rest-frame masks.')

### 7.2 Quality flags used downstream

In [ ]:
data['qc_ok'] = (data['read_error'].isna() & data['nsa_z'].notna() &
                 (data['frac_valid_1RE'] > 0.5) & (data['snr_norm'] > 3) & np.isfinite(data['norm_flux']))
print(data['qc_ok'].value_counts())
display(pd.crosstab(data['broad_type'], data['qc_ok'], normalize='index').round(3))

## 8. Rest-frame stacks by morphological type

In [ ]:
def stack(mask):
    X = REST[mask.values]
    return np.nanmedian(X, axis=0), np.nanpercentile(X, 16, axis=0), np.nanpercentile(X, 84, axis=0), mask.sum()

ok = data['qc_ok']
tt_vals = sorted(data.loc[ok, 'Ttype'].dropna().unique())
cmap = plt.cm.coolwarm_r
fig, ax = plt.subplots(figsize=(15, 5.5))
for i, t in enumerate(tt_vals):
    m = ok & (data['Ttype'] == t)
    if m.sum() < 20: continue
    med, *_ , n = stack(m)
    ax.plot(REST_GRID, med, lw=.8, color=cmap(i / max(1, len(tt_vals) - 1)), label=f'T={int(t)} (n={n})')
ax.set_xlabel('rest λ [Å]'); ax.set_ylabel(f'flux / median{NORM_WINDOW}')
ax.set_title('Median normalised spectrum per T-type'); ax.legend(ncol=4, fontsize=8); ax.set_ylim(0, None)
plt.tight_layout(); plt.show()

## 9. Early- vs late-type galaxies: how the spectra and metadata relate

**Classification used here** (edit it in the Configuration cell):
* **Early type:** `Ttype ≤ 0`, which covers E, Edc, S0, SAB0, SB0 and S0a. These galaxies are dominated by a bulge or spheroid.
* **Late type:** `1 ≤ Ttype ≤ 10` (Sa → Sd/Irr), plus `Ttype = 11` (`Type = 'S'`, a spiral with no sub-type). These galaxies are dominated by a disc.

**What astrophysics leads us to expect**, which the cells below test on this sample:

| Observable | Early types (typically) | Late types (typically) | Source |
|---|---|---|---|
| `D4000n` | Large break (old stars) | Smaller break | spectrum |
| `HdA` | Weak or negative | Stronger (A stars, recent star formation) | spectrum |
| `EW_Ha`, `EW_OII` | ≈0 or weak (LINER-like at most) | Clear emission (ongoing star formation) | spectrum |
| `Mgb`, `Fe`, `NaD` | Strong (old, metal-rich) | Weaker | spectrum |
| `color_br` | Redder | Bluer (but dust reddens edge-on discs) | spectrum |
| `LogMass` | Higher | Lower on average | metadata |
| `incl` | Lower apparent inclination (round light profile) | Full range; edge-on discs reach high values | metadata |
| `R90_kpc` | More compact at fixed mass | Larger at fixed mass | metadata |

Morphology is a **structural** label, while spectra measure **stellar populations**. The two are strongly correlated, but they are not the same thing. Red, passive spirals and blue, star-forming S0s exist. Section 9.8 looks at these "mismatches".

In [ ]:
SPEC_FEATS = ['D4000n_1RE', 'HdA', 'Hb', 'Mgb', 'Fe5270', 'Fe5335', 'NaD', 'EW_OII', 'EW_OIII', 'EW_Ha_1RE', 'color_br']
META_FEATS = ['LogMass', 'nsa_z', 'incl', 'PETRO_TH90', 'R90_kpc']
data['Fe_mean'] = data[['Fe5270', 'Fe5335']].mean(axis=1)
data['dD4000_1RE_minus_FoV'] = data['D4000n_1RE'] - data['D4000n_FOV']
data['dEWHa_1RE_minus_FoV']  = data['EW_Ha_1RE'] - data['EW_Ha_FOV']

ev = data[data['qc_ok'] & data['morph_class'].isin(['early', 'late'])].copy()
print(ev['morph_class'].value_counts())
print('Fraction of galaxies with a measurable index (band coverage >= MIN_BAND_COVERAGE):')
display(ev.groupby('morph_class')[SPEC_FEATS].apply(lambda d: d.notna().mean()).T.round(3))
display(ev.groupby('morph_class')[SPEC_FEATS + META_FEATS].median().T.round(3))

### 9.1 How much does each variable separate early from late on its own?
For each variable we report the **univariate ROC AUC**: 0.5 means no separation and 1 (or 0) means perfect separation. We also report the Mann–Whitney p-value and the Spearman correlation with `Ttype`. This puts the spectral and metadata variables on one common scale.

In [ ]:
rows = []
for c in SPEC_FEATS + ['Fe_mean', 'dD4000_1RE_minus_FoV', 'dEWHa_1RE_minus_FoV'] + META_FEATS:
    d = ev[[c, 'is_early', 'Ttype']].replace([np.inf, -np.inf], np.nan).dropna()
    if d['is_early'].nunique() < 2: continue
    auc = roc_auc_score(d['is_early'], d[c])
    p = stats.mannwhitneyu(d.loc[d.is_early == 1, c], d.loc[d.is_early == 0, c]).pvalue
    rho = stats.spearmanr(d[c], d['Ttype']).correlation
    rows.append(dict(feature=c, source='metadata' if c in META_FEATS else 'spectrum', n=len(d),
                     median_early=d.loc[d.is_early == 1, c].median(), median_late=d.loc[d.is_early == 0, c].median(),
                     AUC_early=auc, separation=abs(auc - 0.5) * 2, mannwhitney_p=p, spearman_vs_Ttype=rho))
uni = pd.DataFrame(rows).sort_values('separation', ascending=False)
display(uni.round(4))

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=uni, y='feature', x='separation', hue='source', dodge=False, ax=ax)
ax.set_xlabel('|2·AUC − 1|  (0 = no separation, 1 = perfect)'); ax.set_title('Univariate early/late separation power')
plt.tight_layout(); plt.show()

### 9.2 Distributions by class and along the Hubble sequence

In [ ]:
show = ['D4000n_1RE', 'HdA', 'EW_Ha_1RE', 'EW_OII', 'Mgb', 'color_br', 'LogMass', 'R90_kpc', 'incl']
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for ax, c in zip(axes.ravel(), show):
    d = ev[[c, 'broad_type']].dropna()
    lo, hi = d[c].quantile([0.01, 0.99])
    sns.violinplot(data=d[(d[c] >= lo) & (d[c] <= hi)], x='broad_type', y=c, order=['E', 'S0/S0a', 'Spiral/Irr'],
                   cut=0, inner='quartile', ax=ax, hue='broad_type', hue_order=['E', 'S0/S0a', 'Spiral/Irr'], palette='coolwarm_r', legend=False)
    ax.set_xlabel('')
plt.suptitle('Spectral and metadata properties by broad morphology', y=1.0); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, c in zip(axes.ravel(), ['D4000n_1RE', 'HdA', 'EW_Ha_1RE', 'Mgb', 'LogMass', 'R90_kpc']):
    d = ev[[c, 'Ttype']].dropna(); lo, hi = d[c].quantile([0.01, 0.99])
    sns.boxplot(data=d[(d[c] >= lo) & (d[c] <= hi)], x='Ttype', y=c, ax=ax, color='lightsteelblue', fliersize=1)
    ax.axvline(list(sorted(d['Ttype'].unique())).index(min(t for t in d['Ttype'].unique() if t > EARLY_TTYPE_MAX)) - .5,
               c='k', ls='--', lw=.8)
    ax.set_title(f'{c} along the Hubble sequence (dashed = early/late cut)')
plt.tight_layout(); plt.show()

### 9.3 Diagnostic planes: spectra × metadata
* **D4000n vs stellar mass.** This is the classic bimodality: a red sequence at high D4000 and a blue cloud at low D4000. The early/late split should follow it closely.
* **EW(Hα) vs D4000n.** This plane separates star-forming from passive galaxies.
* **Colour vs mass.** This is the spectroscopic version of the colour–mass diagram.
* **HδA vs D4000n.** This traces star-formation history. Galaxies offset to high HδA at a given D4000 may have had a recent burst or been quenched recently.

In [ ]:
pairs = [('LogMass', 'D4000n_1RE'), ('D4000n_1RE', 'EW_Ha_1RE'), ('LogMass', 'color_br'), ('D4000n_1RE', 'HdA')]
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
pal = {'early': 'firebrick', 'late': 'royalblue'}
for ax, (x, y) in zip(axes.ravel(), pairs):
    d = ev[[x, y, 'morph_class']].dropna()
    d = d[(d[x].between(*d[x].quantile([.005, .995]))) & (d[y].between(*d[y].quantile([.005, .995])))]
    for cl, g in d.groupby('morph_class'):
        ax.scatter(g[x], g[y], s=3, alpha=.25, c=pal[cl], label=f'{cl} (n={len(g)})')
        try:
            sns.kdeplot(data=sample_n(g, 3000), x=x, y=y, levels=4, color=pal[cl], ax=ax, linewidths=1)
        except Exception: pass
    if y == 'EW_Ha_1RE': ax.set_yscale('symlog', linthresh=1)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.legend(markerscale=4)
plt.tight_layout(); plt.show()

### 9.4 Mass-controlled comparison
Early types are more massive on average, and massive galaxies tend to be old whatever their shape. To find out whether the spectral differences are **tied to morphology** or are just **driven by mass**, we compare the two classes **in narrow mass bins**. We also plot the early-type fraction as a function of mass.

In [ ]:
bins = np.arange(np.floor(ev['LogMass'].quantile(.01) * 4) / 4, np.ceil(ev['LogMass'].quantile(.99) * 4) / 4 + .25, 0.25)
ev['mass_bin'] = pd.cut(ev['LogMass'], bins)
g = ev.groupby(['mass_bin', 'morph_class'], observed=True)
fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
for ax, c in zip(axes[:3], ['D4000n_1RE', 'EW_Ha_1RE', 'HdA']):
    t = g[c].agg(['median', lambda s: s.quantile(.25), lambda s: s.quantile(.75), 'count']).reset_index()
    t.columns = ['mass_bin', 'cls', 'med', 'q25', 'q75', 'n']; t = t[t.n >= 15]
    t['m'] = t['mass_bin'].apply(lambda b: b.mid)
    for cl, tt in t.groupby('cls'):
        ax.plot(tt['m'], tt['med'], 'o-', c=pal[cl], label=cl); ax.fill_between(tt['m'], tt['q25'], tt['q75'], color=pal[cl], alpha=.2)
    ax.set_xlabel('log M*'); ax.set_title(f'{c} at fixed mass'); ax.legend()
fe = ev.groupby('mass_bin', observed=True)['is_early'].agg(['mean', 'count']).reset_index()
fe = fe[fe['count'] >= 15]; fe['m'] = fe['mass_bin'].apply(lambda b: b.mid)
axes[3].plot(fe['m'], fe['mean'], 'ko-'); axes[3].set_ylim(0, 1); axes[3].set_title('Early-type fraction vs mass'); axes[3].set_xlabel('log M*')
plt.tight_layout(); plt.show()

### 9.5 Stacked spectra and *where* in wavelength the classes differ
At each rest-frame wavelength we compute the AUC of the normalised flux between early and late types. Values far from 0.5 mark the parts of the spectrum that carry morphological information, for example the continuum shape, the 4000 Å break and the emission lines.

In [ ]:
okm = ev.index
X_all = REST[okm.values]
# The spectra have many masked pixels (~25% in the sample CSVs), so we keep wavelengths / galaxies with enough
# valid data and fill the remaining holes with an iterative "gappy PCA" reconstruction (not straight-line interpolation).
good_cols = np.mean(np.isfinite(X_all), axis=0) >= PIX_MIN_VALID
good_rows = np.mean(np.isfinite(X_all[:, good_cols]), axis=1) >= ROW_MIN_VALID
Xg = X_all[good_rows][:, good_cols].astype(np.float64)
MISS = ~np.isfinite(Xg)
wav = REST_GRID[good_cols]
y = ev['is_early'].values[good_rows]
ttype_s = ev['Ttype'].values[good_rows]
print('Spectral matrix:', Xg.shape, f'| missing pixels: {MISS.mean():.1%} | early fraction: {y.mean():.3f}')

def gappy_pca_fill(X, miss, n_comp=10, n_iter=8):
    X = X.copy(); col_med = np.nanmedian(X, axis=0)
    X[miss] = np.take(col_med, np.where(miss)[1])
    for _ in range(n_iter):
        mu = X.mean(axis=0); p = PCA(n_components=n_comp, random_state=RANDOM_STATE).fit(X - mu)
        rec = p.inverse_transform(p.transform(X - mu)) + mu
        X[miss] = rec[miss]
    return X
Xs = gappy_pca_fill(Xg, MISS)

# per-pixel AUC using only *observed* pixels (no filled values)
auc_pix = np.full(Xg.shape[1], np.nan)
for j in range(Xg.shape[1]):
    v = ~MISS[:, j]
    if v.sum() > 50 and 0 < y[v].mean() < 1:
        auc_pix[j] = roc_auc_score(y[v], Xg[v, j])
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw=dict(height_ratios=[3, 1.5, 1.5]))
for cl, lab in [(1, 'early'), (0, 'late')]:
    med = np.nanmedian(Xg[y == cl], axis=0); lo, hi = np.nanpercentile(Xg[y == cl], [16, 84], axis=0)
    axes[0].plot(wav, med, c=pal[lab], lw=.8, label=f'{lab} (n={int((y==cl).sum())})'); axes[0].fill_between(wav, lo, hi, color=pal[lab], alpha=.15)
axes[0].legend(); axes[0].set_ylabel('normalised flux'); axes[0].set_title('Median stacks (band = 16–84%)')
axes[1].plot(wav, np.nanmedian(Xg[y == 1], axis=0) / np.nanmedian(Xg[y == 0], axis=0), c='k', lw=.7); axes[1].axhline(1, c='grey', ls=':')
axes[1].set_ylabel('early / late')
axes[2].plot(wav, auc_pix, c='purple', lw=.7); axes[2].axhline(.5, c='grey', ls=':'); axes[2].set_ylabel('per-pixel AUC'); axes[2].set_xlabel('rest λ [Å]')
for a in axes:
    for lam in [3727, 4000, 4102, 4861, 5007, 5175, 5893, 6563]: a.axvline(lam, c='grey', lw=.4, ls=':')
plt.tight_layout(); plt.show()

### 9.6 PCA (and optional UMAP) of the spectra

In [ ]:
pca = PCA(n_components=10, random_state=RANDOM_STATE)
Z = pca.fit_transform(Xs - Xs.mean(axis=0))
print('Explained variance ratio:', pca.explained_variance_ratio_.round(4))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc = axes[0].scatter(Z[:, 0], Z[:, 1], c=ttype_s, cmap='coolwarm_r', s=3, alpha=.6, vmin=-5, vmax=8)
plt.colorbar(sc, ax=axes[0], label='Ttype'); axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].set_title('PCA of normalised spectra')
for i in range(3):
    axes[1].plot(wav, pca.components_[i] + 0.15 * i, lw=.7, label=f'PC{i+1}')
axes[1].legend(); axes[1].set_title('Eigen-spectra (offset)'); axes[1].set_xlabel('rest λ [Å]')
pcd = pd.DataFrame({'PC1': Z[:, 0], 'Ttype': ttype_s})
sns.boxplot(data=pcd, x='Ttype', y='PC1', ax=axes[2], color='lightsteelblue', fliersize=1); axes[2].set_title('PC1 along the Hubble sequence')
plt.tight_layout(); plt.show()
print('AUC of each PC for early vs late:', {f'PC{i+1}': round(roc_auc_score(y, Z[:, i]), 3) for i in range(5)})

In [ ]:
try:
    import umap
    idx = rng.choice(len(Z), size=min(8000, len(Z)), replace=False)
    emb = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=RANDOM_STATE).fit_transform(Z[idx])
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=ttype_s[idx], cmap='coolwarm_r', s=3, vmin=-5, vmax=8)
    plt.colorbar(sc, label='Ttype'); ax.set_title('UMAP of spectral PCA space'); plt.show()
except ImportError:
    print('umap-learn not installed; skipping')

### 9.7 Aperture dependence (1 Rₑ vs full IFU field of view)
Many early types have uniformly old populations, while spirals tend to have older bulges and younger discs. A positive `D4000(1Rₑ) − D4000(FoV)` therefore points to an older centre than outskirts. Keep in mind that the FoV covers a fixed angle on the sky, so how much of the galaxy it includes depends on redshift and size.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, c in zip(axes[:2], ['dD4000_1RE_minus_FoV', 'dEWHa_1RE_minus_FoV']):
    d = ev[[c, 'broad_type']].dropna(); lo, hi = d[c].quantile([.02, .98])
    sns.violinplot(data=d[d[c].between(lo, hi)], x='broad_type', y=c, order=['E', 'S0/S0a', 'Spiral/Irr'], cut=0, ax=ax, hue='broad_type', hue_order=['E', 'S0/S0a', 'Spiral/Irr'], palette='coolwarm_r', legend=False)
    ax.axhline(0, c='k', lw=.6)
d = ev[['D4000n_1RE', 'D4000n_FOV', 'morph_class']].dropna()
for cl, g_ in d.groupby('morph_class'):
    axes[2].scatter(g_['D4000n_FOV'], g_['D4000n_1RE'], s=3, alpha=.3, c=pal[cl], label=cl)
lim = d[['D4000n_1RE', 'D4000n_FOV']].quantile([.01, .99]).values
axes[2].plot([lim.min(), lim.max()], [lim.min(), lim.max()], 'k--', lw=.7); axes[2].set_xlabel('D4000n FoV'); axes[2].set_ylabel('D4000n 1Re'); axes[2].legend(markerscale=4)
plt.tight_layout(); plt.show()

#### Inclination and dust: a common source of confusion
Edge-on discs pass through more dust, so they look redder and can mimic an older population. If late types get redder `color_br` (and, to a lesser degree, higher `D4000n`) as `incl` increases, then some "red spirals" are dust-reddened rather than passive.

In [ ]:
late = ev[ev['morph_class'] == 'late'].copy()
late['incl_bin'] = pd.cut(late['incl'], [0, 30, 45, 60, 70, 80, 90])
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, c in zip(axes, ['color_br', 'D4000n_1RE', 'EW_Ha_1RE']):
    d = late[[c, 'incl_bin']].dropna(); lo, hi = d[c].quantile([.02, .98])
    sns.boxplot(data=d[d[c].between(lo, hi)], x='incl_bin', y=c, ax=ax, color='lightblue', fliersize=1)
    ax.set_title(f'Late types: {c} vs inclination'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

### 9.8 Predictive comparison: metadata vs spectrum vs both
Gradient-boosted trees (which handle NaN natively) are trained with 5-fold stratified CV on four feature sets:
1. **metadata only** (`LogMass`, `nsa_z`, `incl`, `PETRO_TH90`, `R90_kpc`)
2. **spectral indices only**
3. **spectral PCA (10 PCs)**
4. **everything combined**

The gap between these scores shows how much information each source carries about morphology, and whether the two sources complement each other.

*Note on `incl`:* inclination is usually derived from the axis ratio, which is itself a morphological property. So part of what `incl` contributes is shape information rather than independent physics.

In [ ]:
ev2 = ev.iloc[np.where(good_rows)[0]].copy()          # rows that also have a PCA projection
for i in range(10): ev2[f'PC{i+1}'] = Z[:, i]
PCS = [f'PC{i+1}' for i in range(10)]
SPEC_IDX = SPEC_FEATS + ['D4000n_FOV', 'EW_Ha_FOV', 'dD4000_1RE_minus_FoV']
feature_sets = {'metadata': META_FEATS, 'spectral indices': SPEC_IDX, 'spectral PCA': PCS,
                'indices + PCA': SPEC_IDX + PCS, 'all (meta + spectra)': META_FEATS + SPEC_IDX + PCS}
yv = ev2['is_early'].values
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
res, oof = [], {}
for name, cols in feature_sets.items():
    Xf = ev2[cols].replace([np.inf, -np.inf], np.nan).values
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, class_weight='balanced', random_state=RANDOM_STATE)
    p = cross_val_predict(clf, Xf, yv, cv=cv, method='predict_proba')[:, 1]
    oof[name] = p
    res.append(dict(features=name, n_features=len(cols), ROC_AUC=roc_auc_score(yv, p), balanced_acc=balanced_accuracy_score(yv, p > .5)))
res = pd.DataFrame(res); display(res.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, p in oof.items():
    fpr, tpr, _ = roc_curve(yv, p); axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(yv, p):.3f})")
axes[0].plot([0, 1], [0, 1], 'k:'); axes[0].legend(fontsize=8); axes[0].set_title('Out-of-fold ROC: early vs late')
ConfusionMatrixDisplay(confusion_matrix(yv, oof['all (meta + spectra)'] > .5), display_labels=['late', 'early']).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Confusion matrix (all features)'); plt.tight_layout(); plt.show()

In [ ]:
# Which variables matter in the combined model? (permutation importance on a held-out split)
cols = META_FEATS + SPEC_IDX
Xtr, Xte, ytr, yte = train_test_split(ev2[cols].replace([np.inf, -np.inf], np.nan), yv, test_size=.25, stratify=yv, random_state=RANDOM_STATE)
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, class_weight='balanced', random_state=RANDOM_STATE).fit(Xtr, ytr)
pi = permutation_importance(clf, Xte, yte, scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE, n_jobs=N_JOBS)
imp = pd.DataFrame({'feature': cols, 'importance': pi.importances_mean, 'std': pi.importances_std,
                    'source': ['metadata' if c in META_FEATS else 'spectrum' for c in cols]}).sort_values('importance', ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=imp, y='feature', x='importance', hue='source', dodge=False, ax=ax)
ax.set_title('Permutation importance (drop in ROC AUC), meta + indices model'); plt.tight_layout(); plt.show()

### 9.9 Continuous view and "mismatches"
Plotting the out-of-fold probability of being early-type against `Ttype` shows whether the spectra follow a **continuous** Hubble sequence. It also shows where S0/S0a galaxies (the boundary class) end up. The tables then list the most confident mismatches, which are good candidates for visual inspection: **passive/red spirals** and **star-forming early types**.

In [ ]:
ev2['p_early'] = oof['all (meta + spectra)']
fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=ev2, x='Type', y='p_early', order=[t for t in order if t in ev2['Type'].unique()], ax=ax, color='lightsteelblue', fliersize=1)
ax.axhline(.5, c='k', ls='--', lw=.7); ax.tick_params(axis='x', rotation=90); ax.set_title('Predicted P(early) by Hubble Type (out-of-fold)')
plt.tight_layout(); plt.show()

show_cols = ['name', 'Type', 'Ttype', 'LogMass', 'incl', 'D4000n_1RE', 'EW_Ha_1RE', 'HdA', 'color_br', 'p_early']
print('Late-type labels that look passive (high P(early)):')
display(ev2[ev2.is_early == 0].nlargest(10, 'p_early')[show_cols].round(3))
print('Early-type labels that look star-forming (low P(early)):')
display(ev2[ev2.is_early == 1].nsmallest(10, 'p_early')[show_cols].round(3))

In [ ]:
# Visual check of a few mismatches
mm = pd.concat([ev2[ev2.is_early == 0].nlargest(2, 'p_early'), ev2[ev2.is_early == 1].nsmallest(2, 'p_early')])
fig, axes = plt.subplots(len(mm), 1, figsize=(15, 3 * len(mm)))
for ax, (_, r) in zip(np.atleast_1d(axes), mm.iterrows()):
    plot_spectrum(r, ax, show_fov=False); ax.text(.01, .85, f"P(early)={r['p_early']:.2f}", transform=ax.transAxes)
plt.tight_layout(); plt.show()

### 9.10 How to read the results
Fill this in after running the notebook. The cells above give you the evidence for each question:
* **Which spectral observables best track morphology?** See the top rows of the univariate table (9.1). The prior expectation is `D4000n`, `color_br` and `EW_Ha`.
* **Does the link survive at fixed mass?** (9.4) If the early/late curves stay apart within mass bins, morphology carries information beyond mass.
* **How much do the spectra add to the metadata?** Compare the ROC AUC for "metadata" against "all" in 9.8.
* **Is S0/S0a the main source of confusion?** See 9.9. This boundary class usually has the most intermediate probabilities.
* **Do dust or aperture bias the picture?** See the inclination and 1Rₑ-vs-FoV plots (9.7).
* **Data caveats:** masked pixels at emission lines (7.1), the reliability of the FoV errors (7), and rows with sentinel redshifts.

## 10. Export

In [ ]:
out_cols = ['name', 'objra', 'objdec', 'Type', 'Ttype', 'morph_class', 'broad_type', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc',
            'qc_ok', 'n_rows', 'frac_valid_1RE', 'n_gaps', 'snr_1RE', 'snr_FOV', 'snr_norm'] + SPEC_IDX + ['Fe_mean', 'dEWHa_1RE_minus_FoV']
out = data[out_cols].merge(ev2[['name', 'p_early'] + PCS], on='name', how='left')
out_path = Path(CACHE_DIR) / 'manga_eda_galaxy_table.csv'
out.to_csv(out_path, index=False)
np.save(Path(CACHE_DIR) / 'rest_grid.npy', REST_GRID)
print('Saved', out_path, out.shape)
out.head()